### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_mcp_server_fqdn = ! terraform output -raw aca_mcp_server_fqdn
aca_mcp_server_fqdn = aca_mcp_server_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
MCP Server Endpoint: mcp-server.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 4096 # 8736 # 131072 # 512
)

In [5]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I am a large language model, trained by Google.

If you think of me as a digital assistant or a creative collaborator, that’s a good way to put it. I don’t have a physical body, personal feelings, or a life story, but I have been trained on a massive amount of text data, which allows me to process information and communicate in a human-like way.

Here is a breakdown of what I can do and how I function:

### 🛠️ What I can do
*   **Answer Questions:** From complex scientific concepts to "how-to" guides or quick trivia.
*   **Write and Create:** I can draft emails, essays, poems, scripts, stories, and more.
*   **Code and Technical Work:** I can write code in various languages, debug errors, and explain technical documentation.
*   **Summarize:** I can take long articles or documents and boil them down to the key points.
*   **Translate:** I can communicate and translate across dozens of different languages.
*   **Brainstorm:** If you're stuck on a project, I can help generate ideas, outl

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [10]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 3, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 3,
  "results": [
    {
      "title": "LLM Leaderboard 2026 \u2014 Compare 300+ Top AI Models by Intelligence ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "LLM Leaderboard - Comparison of over 100 AI models from OpenAI, Google ...",
      "url": "https://artificialanalysis.ai/leaderboards/models",
      "description": "Comparison and ranking the performance of over 100 AI <b>models</b> (<b>LLMs</b>) across key metrics including intelligence, price, performance and speed (output speed - tokens per second &amp; latency

In [11]:
# Extract first URL from search results
first_url = data["results"][0]["url"]
print("First URL:", first_url)

# Find the tool called "search" by name
fetch_web_content_tool = next(t for t in mcp_tools_web_search if t.name == "fetchWebContent")

# Fetch the page content using the fetch tool
result = await fetch_web_content_tool.ainvoke({"url": first_url, "maxChars": 30000})

# Parse and display the fetched content
web_content_json = json.loads(result[0]["text"])
print(json.dumps(web_content_json, indent=2))

First URL: https://llm-stats.com/
{
  "url": "https://llm-stats.com/",
  "finalUrl": "https://llm-stats.com/",
  "contentType": "text/html; charset=utf-8",
  "title": "LLM Leaderboard 2026 \u2014 Compare 300+ Top AI Models by Intelligence, Speed & Price",
  "retrievalMethod": "request",
  "truncated": false,
  "content": "AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardAI TrendsLLM UpdatesAI NewsBest AI for...LLM Leaderboard \u2014 Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models \u2014 composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1637tok/sLlama 4 Scoutlongest context window10.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo GenerationVideoText-to-SpeechT

In [12]:
from markdownify import markdownify

markdownify(html=web_content_json["content"])

'AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardAI TrendsLLM UpdatesAI NewsBest AI for...LLM Leaderboard — Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models — composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1637tok/sLlama 4 Scoutlongest context window10.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo GenerationVideoText-to-SpeechTTSSpeech-to-TextSTTEmbeddingsEmbeddingsFull leaderboardFull296OpenAll90d30dRankModelLLM StatsReasoningCodingAgentCode ArenaContextSpeedPricing $/MLicense1Claude Mythos PreviewUNRELEASEDAnthropic70.271.357.348.6———$36.11Proprietary2GPT-5.5NEWOpenAI64.162.953.143.61,1021.1M129c/s$7.78Proprietary3Claude Opus 4.7Anthropic61.162.851.642

In [15]:
from IPython.display import display, Markdown

display(Markdown(web_content_json["content"]))  # if fetch_data is a string

AI LeaderboardsLLM LeaderboardOpen LLM LeaderboardAI TrendsLLM UpdatesAI NewsBest AI for...LLM Leaderboard — Compare 300+ Top AI Models by Intelligence, Speed & PriceIndependent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI models — composite LLM Stats Score, updated continuously from public benchmarks and live API metrics.Current leadersLiveClaude Mythos Previewleads on reasoning94.6%gpqaGemini 3.1 Prowins at coding21arenaKimi K2.6cheapest in the top 10$0.95/M tokMercury 2fastest output1637tok/sLlama 4 Scoutlongest context window10.0M tokenstokensKimi K2.6best open-weights90.5%gpqaLLMLLMImage GenerationImageVideo GenerationVideoText-to-SpeechTTSSpeech-to-TextSTTEmbeddingsEmbeddingsFull leaderboardFull296OpenAll90d30dRankModelLLM StatsReasoningCodingAgentCode ArenaContextSpeedPricing $/MLicense1Claude Mythos PreviewUNRELEASEDAnthropic70.271.357.348.6———$36.11Proprietary2GPT-5.5NEWOpenAI64.162.953.143.61,1021.1M129c/s$7.78Proprietary3Claude Opus 4.7Anthropic61.162.851.642.21,8431.0M62c/s$7.22Proprietary4GPT-5.4OpenAI61.158.044.337.71,7171.0M223c/s$3.89Proprietary5GPT-5.2 ProOpenAI61.056.5—29.8—400K—$37.33Proprietary6Kimi K2.6NEWMoonshot AI58.859.545.638.91,229262K31c/s$1.29Open Source7Gemini 3.1 ProGoogle57.859.144.133.82,0931.0M—$3.89Proprietary8Claude Opus 4.6Anthropic57.559.845.638.12,0031.0M106c/s$7.22Proprietary9Seed 2.0 ProByteDance56.854.533.329.2————Proprietary10GPT-5.2OpenAI56.154.235.725.91,516400K191c/s$3.11Proprietary11Gemini 3 ProGoogle56.050.333.424.11,579———Proprietary12Gemini 3 FlashGoogle54.749.831.525.51,6941.0M292c/s$0.78Proprietary13GPT-5.1 InstantOpenAI54.649.731.3—777400K216c/s$2.22Proprietary14GPT-5.1OpenAI53.348.131.8—1,225400K426c/s$2.22Proprietary15GPT-5 MediumOpenAI53.244.3——1,095400K120c/s$2.22Proprietary16Muse SparkMeta53.053.032.925.6————Proprietary17GPT-5.1 HighOpenAI52.052.9——1,140———Proprietary18DeepSeek-V4-Pro-MaxNEWDeepSeek52.057.745.036.65901.0M54c/s$1.93Open Source19GPT-5.1 ThinkingOpenAI51.846.330.8—1,004400K161c/s$2.22Proprietary20Qwen3.6 PlusAlibaba Cloud / Qwen Team51.252.543.331.9—1.0M—$0.78Proprietary1-20 of 296Previous12345NextRecentNew ModelsAnnounced in the last 15 days.All updatesIndexPerformance IndexComposite TrueSkill ratings across published benchmarks.Full leaderboardLeaderboard guideCompare the best AI models with one independent score.The LLM Stats leaderboard ranks GPT, Claude, Gemini, Llama, DeepSeek, Qwen, Mistral, GLM and more by intelligence, speed and price. Every score is sourced from public benchmarks and live API metrics.Read the methodologyHow the LLM Stats Score is computedCompare two modelsPricing, context, speed and benchmark scoresExplore benchmarksMMLU, GPQA, SWE-Bench, AIME and moreFAQQuick answers for choosing, comparing and interpreting today's leading AI models.Which AI model ranks #1 on the LLM Leaderboard?On the LLM Stats Leaderboard, Claude Mythos Preview currently leads on GPQA Diamond (94.6% gpqa), the most discriminating reasoning benchmark at the frontier. This AI leaderboard ranks models by the LLM Stats Score, which aggregates GPQA, SWE-Bench Verified, coding-arena performance and pricing into one comparable AI ranking. Rankings refresh continuously as new benchmark results land.What is the best AI model right now?"Best" depends on what you're optimizing for. For frontier reasoning, Claude Mythos Preview leads on GPQA. For coding agents, Gemini 3.1 Pro is the strongest in head-to-head coding-arena play. For low cost at frontier quality, Kimi K2.6 is the cheapest in the top 10 at $0.95 /M tok. The leaders summary above the table names the current winner per axis.What are the best LLMs in 2026?The leading LLMs in 2026 are Claude Mythos Preview, Gemini 3.1 Pro, and the frontier models from OpenAI (GPT-5 family), Anthropic (Claude Opus and Sonnet), Google (Gemini 3 Pro), xAI (Grok 4), DeepSeek (V3 / R1) and Z.AI (GLM-5). Open-weights leaders include Llama, Qwen and DeepSeek. The full ranking is in the leaderboard table above.What is the cheapest AI model in the top 10?Kimi K2.6 is the cheapest model in the top 10 by GPQA Diamond, at $0.95 /M tok input. The Cheapest filter on the leaderboard restricts to verified, currently-available frontier models — pricing is pulled from each provider's public price list and cross-checked against billing samples through the LLM Stats proxy.Which AI model has the largest context window?Llama 4 Scout currently exposes the largest practical context window at 10.0M tokens tokens. Larger context lets you keep more documents, conversation history and tool traces in a single request. For long-document workloads, also consult the per-model "effective context" notes on each model detail page — providers vary in how well they actually use the upper end of their advertised windows.What is the fastest LLM by output speed?Mercury 2 currently has the highest output throughput at 1637 tok/s. Output speed is measured by routing standardized prompts through each provider's API and averaging tokens-per-second over a 7-day rolling window. Fast inference matters most for streaming chat UIs and agentic loops; for batched async workloads, blended price per 1M tokens is usually the better axis.Which is the best open-source AI model?Kimi K2.6 currently leads among open-weights LLMs (90.5% gpqa). The open-weights ecosystem is dominated by Llama, Qwen, DeepSeek, Mistral, GLM and Gemma. The dedicated Open LLM Leaderboard filters this catalog to models with publicly released weights so you can self-host or fine-tune.How is the LLM Stats Score calculated?The LLM Stats Score is a composite that blends verified benchmark results (GPQA Diamond, SWE-Bench Verified, coding-arena), live performance metrics (output throughput, time-to-first-token) and per-token pricing into one comparable number. Pricing and metadata revalidate hourly; live performance updates on a 7-day rolling average. For the full weighting and refresh cadence see the LLM Stats Score methodology. 296 canonical models are tracked across every major lab and inference provider.

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [16]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools and error handling

In [22]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
# model.max_completion_tokens=131072
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [30]:
from langchain_core.messages import HumanMessage

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""Search the web for the latest news about Azure Container Apps in 2026.
                                          Limit search to 1 results only.
                                          Fetch and analyse web pages one by one.
                                          Use official sources only.
                                          Show your sources in the end.""")]},
    stream_mode="values"
):()

step["messages"][-1].pretty_print()

================================== Ai Message ==================================

Based on the official Microsoft documentation accessed, here is the latest information regarding Azure Container Apps as of 2026:

### Overview
Azure Container Apps remains a serverless platform designed to minimize infrastructure management and cost. It allows developers to run containerized applications without needing to manage server configurations or complex orchestration details.

### Current Capabilities & Use Cases
The platform is primarily used for:
*   **Microservices:** Running distributed applications using **Dapr** for API access.
*   **API Endpoints:** Deploying and managing accessible API services.
*   **Background Processing:** Hosting on-demand, scheduled, or event-based jobs.
*   **Event-Driven Processing:** Running **Azure Functions** using triggers and bindings.

### Key Technical Features
*   **Scaling:** Applications can dynamically scale based on HTTP traffic, CPU/memory load, or an